In [ ]:
import pandas as pd
import plotly.express as px
from typing import Literal
import re

def plot_interactive_probes_wide(
    metrics_csv_path: str,
    probes_csv_path: str,
    metric_prefix: str,
    probe_type: Literal['raw', 'atomic'],
    probe_range: tuple = (0, 5)
):
    """
    Creates an interactive plot from wide-format metric files to visualize
    the performance of individual knowledge probes and their paraphrases.

    Args:
        metrics_csv_path (str):
            Path to the consolidated WIDE-FORMAT CSV file, which should be
            'raw_knowledge_probe_metrics.csv'.
        probes_csv_path (str):
            Path to the original CSV file containing the text of the knowledge probes.
        metric_prefix (str):
            The base name of the metric to plot (e.g., 'raw_knowledge_perplexity'
            or 'raw_knowledge_perplexity_delta'). The function will find and plot
            the base column and all corresponding paraphrase columns.
        probe_type (Literal['raw', 'atomic']):
            Specifies the type of probe to determine how hover text is displayed.
        probe_range (tuple, optional):
            A tuple (start, end) specifying the slice of probes to plot.
            Defaults to (0, 5).
    """
    try:
        metrics_df = pd.read_csv(metrics_csv_path)
        probes_df = pd.read_csv(probes_csv_path)
    except FileNotFoundError as e:
        print(f"Error loading files: {e}")
        return

    # Helper function to wrap text for tooltips
    def wrap_text(text, max_length=80):
        if not isinstance(text, str) or len(text) <= max_length:
            return text
        words = text.split()
        wrapped_lines, current_line = [], ""
        for word in words:
            if len(current_line + " " + word) <= max_length:
                current_line += (" " + word) if current_line else word
            else:
                if current_line: wrapped_lines.append(current_line)
                current_line = word
        if current_line: wrapped_lines.append(current_line)
        return "<br>".join(wrapped_lines)

    # Prepare probe text for hovering
    probes_df['probe_index'] = probes_df.index
    if 'raw' in probe_type:
        probes_df['display_text'] = "PROBE: " + probes_df['raw_knowledge_statement'].apply(wrap_text)
    elif 'paraphrase' in probe_type:
        probes_df['display_text'] = "PROBE: " + probes_df['paraphrase_knowledge_statements'][0].apply(wrap_text)

    else:
        raise ValueError("probe_type must be one of 'raw' or 'atomic'")

    # Merge metrics with probe text
    merged_df = pd.merge(metrics_df, probes_df[['probe_index', 'display_text']], on='probe_index', how='left')

    # Filter for the selected range of probes
    start, end = probe_range
    filtered_df = merged_df[(merged_df['probe_index'] >= start) & (merged_df['probe_index'] < end)].copy()

    if filtered_df.empty:
        print(f"No data found for probe indices in range {probe_range}.")
        return

    # Find all columns related to the specified metric_prefix
    value_cols = [metric_prefix] + [col for col in filtered_df.columns if col.startswith(f"{metric_prefix}_paraphrase_")]
    
    if not any(col in filtered_df.columns for col in value_cols):
        print(f"No metric columns found with prefix '{metric_prefix}' in '{metrics_csv_path}'.")
        print(f"\nAvailable columns are:\n" + "\n".join(list(filtered_df.columns)))
        return
        
    # Use pd.melt to convert wide data to long format for plotting
    id_vars = ['step', 'probe_index', 'section', 'display_text']
    long_df = pd.melt(filtered_df, id_vars=id_vars, value_vars=value_cols, var_name='Probe Version', value_name='Metric Value')

    # Clean up the version names for the legend
    def clean_name(name):
        if name == metric_prefix:
            return 'Raw'
        match = re.search(r'paraphrase_(\d+)', name)
        return f'Paraphrase {match.group(1)}' if match else name
    long_df['Probe Version'] = long_df['Probe Version'].apply(clean_name)

    long_df.dropna(subset=['Metric Value'], inplace=True)
    if long_df.empty:
        print(f"All values for metric '{metric_prefix}' in probe range {probe_range} are NaN.")
        return

    # Create the interactive plot
    title = f'Disaggregated Probes {start}-{end-1} for: {metric_prefix}'
    fig = px.line(
        long_df,
        x='step',
        y='Metric Value',
        color='probe_index',
        line_dash='Probe Version',
        hover_name='probe_index',
        custom_data=['Probe Version', 'section', 'display_text'],
        title=title,
        labels={'step': 'Training Step', 'Metric Value': metric_prefix.replace('_', ' ').title()}
    )
    
    fig.update_traces(hovertemplate=(
        "<b>Probe %{hovertext} (%{customdata[0]})</b><br><br>" +
        "Step: %{x}<br>" +
        f"{metric_prefix}: %{{y:.3f}}<br>" +
        "Section: %{customdata[1]}<br>" +
        "%{customdata[2]}" +
        "<extra></extra>"
    ))
    
    fig.update_layout(
        legend_title_text='Probe Index / Version',
        hoverlabel=dict(bgcolor="white", bordercolor="black", font_size=12, font_family="Arial", align="left")
    )
    fig.show()

In [9]:

# --- USAGE EXAMPLE ---

# Configure the paths and parameters below to match your experiment.
RESULTS_DIR = '../../results/FT/SingleArxivPaper_1B_Test_Run'
PROBES_CSV = '../../data/arxiv/DPO_knowledge_probes_v3.csv' # Make sure this is the correct probe file
METRICS_CSV = f'{RESULTS_DIR}/raw_knowledge_probe_metrics.csv'

# Call the function with the metric you want to investigate
plot_interactive_probes_wide(
    metrics_csv_path=METRICS_CSV,
    probes_csv_path=PROBES_CSV,
    metric_prefix='raw_knowledge_perplexity', # Or 'raw_knowledge_perplexity'
    probe_type='raw',
    probe_range=(7,10) # Display probes 0 through 4
)

In [ ]:
plot_interactive_probes_wide(
    metrics_csv_path=METRICS_CSV,
    probes_csv_path=PROBES_CSV,
    metric_prefix='raw_knowledge_perplexity',
    probe_type='raw',
    probe_range=(5, 10) # Display probes 5 through 9
)